# Mathematical Modelling and Simulation of Quantum Gate Operations Using Qiskit

**A reproducible computational research project**  
**Team:** [DAISI WILLIAMS OLOLADE] and [OLUYEDE TITILAYO OLUFUNKE]  
**Date:** October 2026  
**Scope:** Single-qubit Pauli, Hadamard, and rotation gates; sequential gate operations; ideal and finite-shot simulation.

> **Scientific integrity:** This notebook generates results when executed. No unexecuted code or anticipated outcome should be reported as an observed result. All computations are performed on classical simulators; no quantum hardware results are claimed.

## Abstract
We derive and numerically verify the matrix action of fundamental single-qubit gates, visualize state evolution on the Bloch sphere, investigate angle-dependent measurement probabilities, test gate-order dependence, and compare ideal probabilities with finite-shot sampling. The analysis uses independent NumPy matrix calculations as reference and Qiskit statevector and shot-based simulations as computational implementations. Agreement is quantified using phase-invariant state fidelity, Bloch-vector discrepancy, and probability errors.

## Research questions
1. Do Qiskit statevectors reproduce analytically calculated gate transformations?
2. How do $R_x(\theta)$, $R_y(\theta)$ and $R_z(\theta)$ change Bloch coordinates and measurement probabilities?
3. Which gate sequences are order dependent, and can computational-basis measurements distinguish their outputs?
4. How do finite-shot probability estimates converge toward ideal probabilities?

**Pre-specified checks:** unitarity residuals near machine precision, ideal state fidelity near 1, and shot errors consistent with binomial sampling. These are expectations to test, not results.

## 1. Environment and reproducibility
Run the installation cell once in Colab or a fresh Jupyter environment. Restart the kernel if your environment requests it. The following cells record software versions, fix a random seed, and create an output folder. Execute the notebook from top to bottom.

In [ ]:
# Run once if packages are not already installed (uncomment in a local environment).
%pip -q install "qiskit>=1.2,<3" "qiskit-aer>=0.15,<1" numpy scipy pandas matplotlib

In [ ]:
from pathlib import Path
import sys, platform, importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binom
from IPython.display import display
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator, state_fidelity
from qiskit.visualization import plot_bloch_multivector
from qiskit_aer import AerSimulator

SEED = 2026
RNG = np.random.default_rng(SEED)
OUT = Path('quantum_gate_project_results')
OUT.mkdir(exist_ok=True)
for package in ['qiskit', 'qiskit-aer', 'numpy', 'scipy', 'pandas', 'matplotlib']:
    print(f'{package}: {metadata.version(package)}')
print('Python:', sys.version.split()[0], '| OS:', platform.platform())
print('Random seed:', SEED)
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 180})

## 2. Mathematical model
For a pure qubit, $|\psi\rangle=\alpha|0\rangle+\beta|1\rangle$ with $|\alpha|^2+|\beta|^2=1$. A gate $U$ acts by $|\psi'\rangle=U|\psi\rangle$ and must satisfy $U^\dagger U=I$. The Bloch coordinates are $\mathbf r=(\langle X\rangle,\langle Y\rangle,\langle Z\rangle)$. For a statevector $[\alpha,\beta]^T$:

$$r_x=2\operatorname{Re}(\alpha^*\beta),\quad r_y=2\operatorname{Im}(\alpha^*\beta),\quad r_z=|\alpha|^2-|\beta|^2.$$

**Important:** A global phase $e^{i\gamma}$ does not alter physical predictions. Therefore compare statevectors by fidelity, not naive componentwise differences. Relative phase is observable after suitable interference operations.

In [ ]:
I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
GATES = {'I': I, 'X': X, 'Y': Y, 'Z': Z, 'H': H}
ket0 = np.array([1, 0], dtype=complex)
ket1 = np.array([0, 1], dtype=complex)
ket_plus = H @ ket0
ket_plus_i = (ket0 + 1j * ket1) / np.sqrt(2)

def bloch(psi):
    psi = np.asarray(psi, dtype=complex).reshape(2)
    return np.array([2*np.real(np.conj(psi[0])*psi[1]),
                     2*np.imag(np.conj(psi[0])*psi[1]),
                     abs(psi[0])**2 - abs(psi[1])**2], dtype=float)

def probabilities(psi):
    return np.abs(np.asarray(psi, dtype=complex))**2

def fidelity(a, b):
    return float(abs(np.vdot(a, b))**2)

unitarity = pd.DataFrame([{'gate': name,
    '||U†U-I||_F': np.linalg.norm(U.conj().T @ U - I)}
    for name, U in GATES.items()])
display(unitarity)
assert (unitarity['||U†U-I||_F'] < 1e-12).all()
unitarity.to_csv(OUT / 'unitarity_checks.csv', index=False)

## 3. Fundamental gates: analytical predictions versus Qiskit
The experiment includes multiple initial states to reveal phase changes invisible when only |0⟩ is used. Qiskit statevectors are obtained from gate circuits **without measurement**. Analytical outputs are computed independently using NumPy matrices.

In [ ]:
INPUTS = {'|0>': ket0, '|1>': ket1, '|+>': ket_plus, '|+i>': ket_plus_i}

def make_gate_circuit(gate, initial):
    qc = QuantumCircuit(1)
    qc.initialize(initial, 0)
    getattr(qc, gate.lower())(0)
    return qc

records = []
for input_name, initial in INPUTS.items():
    for gate_name in ['X', 'Y', 'Z', 'H']:
        predicted = GATES[gate_name] @ initial
        qc = make_gate_circuit(gate_name, initial)
        observed = Statevector.from_instruction(qc).data
        rec = {'input': input_name, 'gate': gate_name,
               'fidelity': fidelity(predicted, observed),
               'probability_L1_error': np.sum(abs(probabilities(predicted)-probabilities(observed))),
               'bloch_L2_error': np.linalg.norm(bloch(predicted)-bloch(observed)),
               'P0_theory': probabilities(predicted)[0],
               'P0_qiskit': probabilities(observed)[0],
               'P1_theory': probabilities(predicted)[1],
               'P1_qiskit': probabilities(observed)[1]}
        records.append(rec)
results_gates = pd.DataFrame(records)
display(results_gates.round(12))
results_gates.to_csv(OUT / 'fundamental_gate_verification.csv', index=False)
assert np.allclose(results_gates['fidelity'], 1, atol=1e-12)
assert results_gates['probability_L1_error'].max() < 1e-12
print('Example circuit:')
print(make_gate_circuit('H', ket0).draw(output='text'))

### Visualize representative state transformations
The Bloch sphere displays physical state changes; global phases do not move the Bloch vector.

In [ ]:
examples = [('X', ket0, '|0>'), ('Y', ket_plus, '|+>'),
            ('Z', ket_plus, '|+>'), ('H', ket0, '|0>')]
for name, initial, label in examples:
    final = Statevector.from_instruction(make_gate_circuit(name, initial))
    fig = plot_bloch_multivector(final)
    fig.suptitle(f'{name} applied to {label}', y=1.02)
    fig.savefig(OUT / f'bloch_{name}_{label.replace("|", "").replace(">", "")}.png', bbox_inches='tight')
    plt.show()

## 4. Rotation gates: derivation and parameter sweep

$$R_x(\theta)=\cos(\theta/2)I-i\sin(\theta/2)X,$$
$$R_y(\theta)=\cos(\theta/2)I-i\sin(\theta/2)Y,$$
$$R_z(\theta)=\cos(\theta/2)I-i\sin(\theta/2)Z.$$

For input $|0\rangle$, $P_1(R_x)=P_1(R_y)=\sin^2(\theta/2)$, while $P_1(R_z)=0$. Yet $R_z$ is **not** trivial: it changes relative phase for superposition inputs. We therefore also examine $|+\rangle$.

In [ ]:
def rotation_matrix(axis, theta):
    pauli = {'x': X, 'y': Y, 'z': Z}[axis]
    return np.cos(theta/2)*I - 1j*np.sin(theta/2)*pauli

def rotation_circuit(axis, theta, initial):
    qc = QuantumCircuit(1)
    qc.initialize(initial, 0)
    getattr(qc, 'r'+axis)(float(theta), 0)
    return qc

angles = np.linspace(0, 2*np.pi, 73)
rotation_rows = []
for input_name, initial in {'|0>':ket0, '|+>':ket_plus}.items():
    for axis in 'xyz':
        for theta in angles:
            analytic = rotation_matrix(axis, theta) @ initial
            sim = Statevector.from_instruction(rotation_circuit(axis, theta, initial)).data
            rotation_rows.append({'input': input_name, 'axis': axis, 'theta_rad': theta,
                'theta_deg': np.degrees(theta), 'P1_theory': probabilities(analytic)[1],
                'P1_qiskit': probabilities(sim)[1], 'fidelity': fidelity(analytic, sim),
                'rx': bloch(sim)[0], 'ry': bloch(sim)[1], 'rz': bloch(sim)[2]})
rotation_df = pd.DataFrame(rotation_rows)
rotation_df.to_csv(OUT / 'rotation_parameter_sweep.csv', index=False)
rotation_df['abs_prob_error'] = abs(rotation_df.P1_theory-rotation_df.P1_qiskit)
display(rotation_df.groupby(['input','axis']).agg(min_fidelity=('fidelity','min'),
    max_abs_prob_error=('abs_prob_error', 'max')))
print('Maximum probability discrepancy:', np.max(abs(rotation_df.P1_theory-rotation_df.P1_qiskit)))
assert np.min(rotation_df.fidelity) > 1-1e-12

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3), sharey=True)
for ax, initial in zip(axes, ['|0>', '|+>']):
    for axis in 'xyz':
        sub = rotation_df[(rotation_df.input == initial) & (rotation_df.axis == axis)]
        ax.plot(sub.theta_deg, sub.P1_qiskit, label=f'R{axis} Qiskit')
        ax.plot(sub.theta_deg[::6], sub.P1_theory[::6], '.', ms=3, alpha=0.7)
    ax.set(xlabel='Rotation angle (degrees)', title=f'Initial state {initial}', ylim=(-0.05,1.05))
    ax.grid(alpha=0.25); ax.legend()
axes[0].set_ylabel('Probability of outcome 1')
fig.suptitle('Rotation gates: solid lines = Qiskit; dots = analytical samples')
fig.tight_layout(); fig.savefig(OUT/'rotation_probabilities.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
for ax, axis in zip(axes, 'xyz'):
    sub = rotation_df[(rotation_df.input == '|+>') & (rotation_df.axis == axis)]
    for coordinate in ['rx','ry','rz']:
        ax.plot(sub.theta_deg, sub[coordinate], label=coordinate)
    ax.set(title=f'R{axis} acting on |+>', xlabel='Angle (degrees)', ylim=(-1.1,1.1))
    ax.grid(alpha=0.25)
axes[0].set_ylabel('Bloch coordinate')
axes[-1].legend(loc='lower right')
fig.tight_layout(); fig.savefig(OUT/'bloch_coordinate_evolution.png', bbox_inches='tight')
plt.show()

## 5. Gate order and noncommutativity
If $A$ is applied first and $B$ second, the output is $BA|\psi\rangle$. Matrix products are evaluated in reverse chronological order. The commutator is $[A,B]=AB-BA$.

**Measurement caveat:** Different states can have identical $Z$-basis probabilities. We compare state fidelity and Bloch vectors as well as $P(0)$ and $P(1)$.

In [ ]:
def sequence_circuit(initial, sequence):
    qc = QuantumCircuit(1)
    qc.initialize(initial, 0)
    for gate in sequence:
        getattr(qc, gate.lower())(0)
    return qc

def analytic_sequence(initial, sequence):
    result = initial.copy()
    for gate in sequence:
        result = GATES[gate] @ result
    return result

sequence_pairs = [(('H','Z'), ('Z','H')),
                  (('X','H'), ('H','X')),
                  (('H','Z','H'), ('X',))]
seq_rows = []
for a, b in sequence_pairs:
    for initial_name, initial in {'|0>':ket0, '|+>':ket_plus}.items():
        a_sim = Statevector.from_instruction(sequence_circuit(initial,a)).data
        b_sim = Statevector.from_instruction(sequence_circuit(initial,b)).data
        a_math = analytic_sequence(initial,a)
        b_math = analytic_sequence(initial,b)
        seq_rows.append({'input':initial_name, 'sequence_A':' -> '.join(a),
                         'sequence_B':' -> '.join(b),
                         'A_vs_B_fidelity':fidelity(a_sim,b_sim),
                         'A_vs_B_P1_difference':abs(probabilities(a_sim)[1]-probabilities(b_sim)[1]),
                         'A_math_fidelity':fidelity(a_math,a_sim),
                         'B_math_fidelity':fidelity(b_math,b_sim)})
seq_df = pd.DataFrame(seq_rows)
display(seq_df.round(10))
seq_df.to_csv(OUT/'gate_sequence_comparisons.csv',index=False)
print('Commutator Frobenius norm ||HZ-ZH||:', np.linalg.norm(H@Z-Z@H))
print('Example H then Z circuit:')
print(sequence_circuit(ket0, ('H','Z')).draw(output='text'))
assert min(seq_df.A_math_fidelity.min(),seq_df.B_math_fidelity.min()) > 1-1e-12

## 6. Finite-shot measurement experiment
Statevector probabilities are exact up to floating-point error. Counts from $N$ shots are random: $K\sim\operatorname{Binomial}(N,p)$ and $\hat p=K/N$. The predicted standard error is $\sqrt{p(1-p)/N}$.

We estimate $P(1)$ for $H|0\rangle$ over different shot counts. A **single run** need not show monotonic error reduction; the expected uncertainty decreases as $N^{-1/2}$. Repeated experiments provide a more reliable check.

In [ ]:
shot_sizes = [100, 500, 1000, 5000, 10000]
repeats = 40
simulator = AerSimulator(seed_simulator=SEED)
measured = QuantumCircuit(1)
measured.h(0)
measured.measure_all()
p_exact = 0.5
shot_rows = []
for shots in shot_sizes:
    for repetition in range(repeats):
        # Explicit unique seed makes independent runs reproducible.
        result = simulator.run(measured, shots=shots,
                               seed_simulator=SEED+1000*shot_sizes.index(shots)+repetition).result()
        counts = result.get_counts()
        estimate = counts.get('1',0)/shots
        shot_rows.append({'shots':shots,'replicate':repetition,'P1_estimate':estimate,
                          'abs_error':abs(estimate-p_exact),
                          'theoretical_SE':np.sqrt(p_exact*(1-p_exact)/shots)})
shot_df = pd.DataFrame(shot_rows)
shot_summary = shot_df.groupby('shots').agg(
    empirical_RMSE=('P1_estimate',lambda x: np.sqrt(np.mean((x-p_exact)**2))),
    mean_absolute_error=('abs_error','mean'),
    theoretical_SE=('theoretical_SE','first')).reset_index()
display(shot_summary)
shot_df.to_csv(OUT/'shot_measurements_raw.csv',index=False)
shot_summary.to_csv(OUT/'shot_measurement_summary.csv',index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(7,4.5))
ax.loglog(shot_summary.shots, shot_summary.empirical_RMSE, 'o-', label='Empirical RMSE (40 runs)')
ax.loglog(shot_summary.shots, shot_summary.theoretical_SE, 's--', label='Binomial theoretical SE')
ax.set(xlabel='Number of shots', ylabel='Probability estimation error',
       title='Finite-shot convergence for H|0>')
ax.grid(True, which='both', alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig(OUT/'shot_convergence.png', bbox_inches='tight')
plt.show()

## 7. Consolidated verification and interpretation

The following summary reports **computed values** only after previous cells execute. We assess fidelity and probability agreement with fixed tolerances, while finite-shot statistics are interpreted probabilistically, not as deterministic pass/fail guarantees.

In [ ]:
summary = pd.DataFrame([
    {'metric':'Maximum gate unitarity residual', 'value':float(unitarity['||U†U-I||_F'].max())},
    {'metric':'Minimum fundamental-gate fidelity', 'value':float(results_gates.fidelity.min())},
    {'metric':'Maximum fundamental probability L1 error', 'value':float(results_gates.probability_L1_error.max())},
    {'metric':'Minimum rotation fidelity', 'value':float(rotation_df.fidelity.min())},
    {'metric':'Maximum rotation probability error', 'value':float(np.max(abs(rotation_df.P1_theory-rotation_df.P1_qiskit)))},
    {'metric':'Minimum sequence theory/simulation fidelity', 'value':float(min(seq_df.A_math_fidelity.min(),seq_df.B_math_fidelity.min()))},
])
display(summary)
summary.to_csv(OUT/'verification_summary.csv', index=False)
assert summary.loc[summary.metric=='Maximum gate unitarity residual','value'].iloc[0] < 1e-12
assert results_gates.fidelity.min() > 1-1e-12
assert rotation_df.fidelity.min() > 1-1e-12
print('Deterministic verification checks passed.')

## 8. Discussion prompts (complete after running)
1. Which operations change computational-basis populations and which change only relative phase for the chosen input?
2. Why does $R_z(\theta)|0\rangle$ have constant $P(1)$ even though the statevector changes by a phase?
3. Identify a pair of gate sequences that differ in state fidelity but not in $Z$-basis measurement probabilities. What extra measurement basis could distinguish them?
4. Does empirical finite-shot RMSE track the predicted $N^{-1/2}$ trend? Explain any deviations with finite repetitions.
5. What is the difference between numerical verification of known identities and evidence of quantum computational advantage?

## 9. Limitations and possible extensions
- Single-qubit circuits only; no entanglement or quantum advantage is demonstrated.
- Ideal statevectors omit hardware errors; shot simulations model sampling statistics, not device noise.
- Fixed initial states and gate families; findings are not automatically generalizable to arbitrary quantum channels.
- Optional extension: add amplitude damping or depolarizing noise, then compare fidelity with ideal predictions.

## 10. Conclusion template
After execution, summarize the largest unitarity residual, lowest state fidelity, largest probability discrepancy, one notable phase-sensitive result, and the finite-shot scaling trend. Avoid claims of measured speedup or hardware performance.

## References
1. Nielsen, M. A. and Chuang, I. L. *Quantum Computation and Quantum Information*, Cambridge University Press, 2010.
2. Javadi-Abhari, A. et al. *Quantum computing with Qiskit*, arXiv:2405.08810, 2024. https://arxiv.org/abs/2405.08810
3. IBM Quantum, Qiskit documentation: https://quantum.cloud.ibm.com/docs
4. Qiskit Aer documentation: https://qiskit.github.io/qiskit-aer/

## Submission checklist
- [ ] Replace author placeholders and enter project dates.
- [ ] Run **Kernel → Restart & Run All** without errors.
- [ ] Explain each figure and table in the report; do not submit unexplained outputs.
- [ ] Verify CSV and PNG files in the results directory.
- [ ] Record environment versions and cite software documentation.
- [ ] Submit the executed notebook, figures, CSV outputs, README, and project report.

## 11. Export instructions
The following cell creates a compressed results archive for convenient submission. In Google Colab, download the ZIP from the Files pane. In local Jupyter, find it next to this notebook.

In [ ]:
import shutil
archive = shutil.make_archive('quantum_gate_project_results', 'zip', root_dir='.', base_dir=str(OUT))
print('Saved:', archive)
print('Generated files:', len(list(OUT.iterdir())))